# 01 — Hello, memory

A five-minute tour of memtomem's Python API. By the end of this notebook you
will have:

1. Initialised the core components (`storage`, `embedder`, `index_engine`,
   `search_pipeline`) against an isolated temp directory.
2. Added a small handful of markdown memories.
3. Run a hybrid BM25 + dense search and looked at the structure of a result.

No state lands in your real `~/.memtomem/` — everything happens inside a
`tempfile.TemporaryDirectory()` that is removed in the last cell.

## Prerequisites

- **memtomem** with the ONNX extra — local dense embeddings, **no server to
  run**:
  ```bash
  # From PyPI
  uv pip install "memtomem[onnx]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[onnx]" jupyter ipykernel
  ```

That's the whole setup — there is no embedding server to start. The first
search downloads the `bge-small-en-v1.5` model (~67 MB) into
`~/.memtomem/cache/fastembed/` and reuses it on every later run.

In [1]:
# Confirm the ONNX embedding backend is importable before doing anything else.
# fastembed runs the model locally in-process, so there is no server to check.
try:
    import fastembed  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError(
        "fastembed is not installed. Install the ONNX extra and re-run this cell:\n"
        '  uv pip install "memtomem[onnx]" jupyter ipykernel'
    ) from e

print("ONNX embedding backend available.")

ONNX embedding backend available.


## Step 1 — Initialise components in a temp directory

This setup block is the same in every notebook in this folder. Re-use it as
a template when you want to try memtomem in your own scripts.

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables. Note the double underscore —
#    pydantic-settings uses "__" as the nesting separator for section fields.
#    The ONNX provider runs locally via fastembed (no server); "bge-small-en-v1.5"
#    is the short alias for BAAI/bge-small-en-v1.5 (384-dim, ~67 MB).
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "onnx"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "bge-small-en-v1.5"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "384"

# 3. Apply the same fields directly on the config object, and temporarily
#    disable load_config_overrides() so the real ~/.memtomem/config.json
#    cannot leak into this notebook. This mirrors the pattern used in
#    packages/memtomem/tests/conftest.py.
config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"memtomem components ready. db={db_path}")

memtomem components ready. db=/tmp/tmp401a36hs/memory.db


## Step 2 — Add a few memories

We write three tiny markdown files into the temp memory directory and index
them with `index_engine.index_file()`. Each call returns an `IndexingStats`
dataclass so you can see how many chunks were produced.

In [3]:
sample_notes = {
    "python-lists.md": """# Python list comprehensions

`[x ** 2 for x in range(10)]` produces the squares from 0 to 81. Add a filter
with `if`: `[x for x in range(20) if x % 3 == 0]`.
""",
    "deployment.md": """# Deployment checklist

Always run the migration dry-run against staging before promoting to prod.
Rollbacks are cheap when the migration is idempotent.
""",
    "docker-compose.md": """# docker compose notes

Use `docker compose up -d` for detached mode. Since Compose v2 the command is
`docker compose` with a space, not the old `docker-compose` hyphenated form.
""",
}

for filename, content in sample_notes.items():
    (notes_dir / filename).write_text(content, encoding="utf-8")

for filename in sample_notes:
    stats = await comp.index_engine.index_file(notes_dir / filename)
    print(
        f"{filename}: indexed {stats.indexed_chunks} chunks "
        f"(total {stats.total_chunks}) in {stats.duration_ms:.1f} ms"
    )

[06/27/26 12:32:51] INFO     Loading ONNX embedding model BAAI/bge-small-en-v1.5 (threads=4,             ]8;id=263539;file:///Users/you/memtomem/packages/memtomem/src/memtomem/embedding/onnx.py\onnx.py]8;;\:]8;id=876225;file:///Users/you/memtomem/packages/memtomem/src/memtomem/embedding/onnx.py#81\81]8;;\
                             cache_dir=/Users/you/.memtomem/cache/fastembed) …                                

python-lists.md: indexed 1 chunks (total 1) in 63.0 ms
deployment.md: indexed 1 chunks (total 1) in 4.8 ms
docker-compose.md: indexed 1 chunks (total 1) in 5.1 ms


## Step 3 — Run a hybrid search

`search_pipeline.search()` returns a `(results, stats)` tuple. Each
`SearchResult` wraps a `Chunk` plus the fused score and a rank. The `source`
field tells you which retriever the hit came from (`bm25`, `dense`, `fused`,
or `reranked`).

In [4]:
results, stats = await comp.search_pipeline.search(
    query="deployment checklist",
    top_k=5,
)

print(f"bm25 candidates: {stats.bm25_candidates}")
print(f"dense candidates: {stats.dense_candidates}")
print(f"final results: {stats.final_total}\n")

for r in results:
    heading = " / ".join(r.chunk.metadata.heading_hierarchy) or "(no heading)"
    source_file = Path(r.chunk.metadata.source_file).name
    snippet = r.chunk.content[:120].replace("\n", " ")
    print(f"[{r.rank}] score={r.score:.3f} source={r.source:<6} file={source_file} — {heading}")
    print(f"    {snippet}...\n")

bm25 candidates: 1
dense candidates: 3
final results: 3

[1] score=0.033 source=fused  file=deployment.md — # Deployment checklist
    Always run the migration dry-run against staging before promoting to prod. Rollbacks are cheap when the migration is ide...

[2] score=0.016 source=dense  file=docker-compose.md — # docker compose notes
    Use `docker compose up -d` for detached mode. Since Compose v2 the command is `docker compose` with a space, not the old...

[3] score=0.016 source=dense  file=python-lists.md — # Python list comprehensions
    `[x ** 2 for x in range(10)]` produces the squares from 0 to 81. Add a filter with `if`: `[x for x in range(20) if x % 3...



## Step 4 — Inspect one result more closely

The full `SearchResult` / `Chunk` object exposes everything you might need
for downstream processing: the chunk UUID, tags, namespace, heading
hierarchy, and the raw `content` string.

In [5]:
top = results[0]
print("chunk id:        ", top.chunk.id)
print("score (fused):   ", round(top.score, 4))
print("retriever source:", top.source)
print("source file:     ", top.chunk.metadata.source_file)
print("heading path:    ", top.chunk.metadata.heading_hierarchy)
print("tags:            ", top.chunk.metadata.tags)
print("namespace:       ", top.chunk.metadata.namespace)
print()
print("content:")
print(top.chunk.content)

chunk id:         84964643-fc02-4417-9e1a-13f0cfeabd7b
score (fused):    0.0328
retriever source: fused
source file:      /tmp/tmp401a36hs/notes/deployment.md
heading path:     ('# Deployment checklist',)
tags:             ()
namespace:        default

content:
Always run the migration dry-run against staging before promoting to prod.
Rollbacks are cheap when the migration is idempotent.


## Mapping to MCP tools

When the same operations are invoked from an MCP client (Claude Code,
Cursor, Windsurf, …), the equivalents are:

| Python API | MCP tool |
|------------|----------|
| `index_engine.index_file(path)` | `mem_index(path=...)` |
| `search_pipeline.search(query)` | `mem_search(query=...)` |
| `storage.*` mixin methods | `mem_do(action=..., params=...)` |

See [Getting Started → First use](../../docs/guides/getting-started.md#first-use)
for the same flow driven from your AI editor instead of Python.

## Cleanup

In [6]:
# Release connections and remove the temp directory.
await close_components(comp)
tmp.cleanup()

# Clean up the environment variables we set.
for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
